In [4]:
ls /media/notebooks/meddocan_anonimizados.csv

/media/notebooks/meddocan_anonimizados.csv


In [9]:
!hdfs dfs -mkdir -p /datasets/meddocan/resultados/estadisticas_documentos
!hdfs dfs -mkdir -p /datasets/meddocan/csv
!hdfs dfs -put  /media/notebooks/meddocan_anonimizados.csv /datasets/meddocan/csv

!hdfs dfs -ls /datasets/meddocan/csv

Found 1 items
-rw-r--r--   3 root supergroup      48161 2026-10-29 08:44 /datasets/meddocan/csv/meddocan_anonimizados.csv


In [82]:
%%writefile mapper_meddocan.py

#!/usr/bin/env python3

# Importar las bibliotecas necesarias:
# sys permite leer la entrada estándar y escribir mensajes de error.
# csv permite leer correctamente los registros del archivo CSV.
import sys
import csv


# Definir los índices de las columnas del CSV.
# Los índices comienzan en cero, por lo que la columna 12 tiene índice 11.
INDICE_EDAD = 11
INDICE_ESPECIALIDAD = 14

# Número de columnas que debe contener cada registro del archivo CSV.
NUM_COLUMNAS = 24


def clasificar_edad(edad):
    """
    Clasifica una edad en un grupo para facilitar el análisis estadístico.
    Si la edad no es válida, devuelve EDAD_DESCONOCIDA.
    """
    try:
        # Convertir la edad a número entero.
        # Se utiliza float previamente para admitir valores como "40.0".
        edad = int(float(edad.strip()))

        # Asignar la edad al intervalo correspondiente.
        if edad < 18:
            return "menor_18"
        elif edad <= 29:
            return "18_29"
        elif edad <= 44:
            return "30_44"
        elif edad <= 59:
            return "45_59"
        elif edad <= 74:
            return "60_74"
        else:
            # Agrupar las edades de 75 años o más.
            return "75_o_mas"

    except (ValueError, AttributeError):
        # Gestionar las edades vacías o que no se puedan convertir.
        return "EDAD_DESCONOCIDA"


def normalizar_especialidad(especialidad):
    """
    Normaliza el campo de especialidad.
    Si está vacío o contiene únicamente espacios, asigna
    la categoría SIN_ESPECIALIDAD.
    """
    if not especialidad or not especialidad.strip():
        return "SIN_ESPECIALIDAD"

    # Eliminar los espacios al principio y al final del texto.
    return especialidad.strip()


def procesar_fila(fila):
    """
    Procesa un registro del CSV y genera una clave compuesta
    por la especialidad médica y el grupo de edad.
    """

    # Comprobar que el registro contiene el número esperado de columnas.
    # Si no es correcto, informar del problema y descartar la fila.
    if len(fila) != NUM_COLUMNAS:
        print(
            f"Fila descartada: contiene {len(fila)} columnas",
            file=sys.stderr
        )
        return

    # Ignorar la primera fila si corresponde a la cabecera del CSV.
    if fila[0].strip().lower() == "id":
        return

    # Extraer la edad y la especialidad utilizando sus índices.
    edad = fila[INDICE_EDAD]
    especialidad = fila[INDICE_ESPECIALIDAD]

    # Clasificar la edad y normalizar la especialidad.
    grupo_edad = clasificar_edad(edad)
    especialidad = normalizar_especialidad(especialidad)

    # Construir una clave compuesta para relacionar ambas variables.
    # Ejemplo: Oncología|30_44
    clave = f"{especialidad}|{grupo_edad}"

    # Emitir la clave y el valor 1 separados por un tabulador.
    # Hadoop Streaming utiliza este formato para transmitir los
    # registros del mapper al proceso de ordenación y al reducer.
    # El reducer sumará los valores asociados a cada clave.
    print(f"{clave}\t1")


def main():
    """
    Lee el CSV desde la entrada estándar y procesa cada registro.
    Este mecanismo permite recibir los datos desde Hadoop Streaming
    sin necesidad de abrir directamente un archivo local.
    """

    # Crear un lector CSV que interprete correctamente los campos
    # delimitados por comas, incluidos los que contienen comas entre comillas.
    lector = csv.reader(sys.stdin, delimiter=",")

    # Recorrer todos los registros recibidos por la entrada estándar.
    for fila in lector:
        try:
            # Procesar cada fila y generar su correspondiente clave.
            procesar_fila(fila)

        except Exception as error:
            # Enviar los errores a stderr para no mezclarlos con los
            # pares clave-valor que Hadoop Streaming procesa.
            print(
                f"Error al procesar una fila: {error}",
                file=sys.stderr
            )


# Ejecutar la función principal cuando se inicia este archivo.
if __name__ == "__main__":
    main()

Overwriting mapper_meddocan.py


In [84]:
%%writefile reducer_meddocan.py
#!/usr/bin/env python3
import sys

# Diccionario para almacenar el total de cada categoría.
totales = {}

# Recibir los resultados intermedios del mapper.
for linea in sys.stdin:

    linea = linea.rstrip("\n")

    # Separar la clave del valor mediante el tabulador.
    try:
        clave, valor = linea.split("\t", 1)
        cantidad = int(valor)
    except ValueError:
        continue

    # Acumular el número de registros de cada categoría.
    totales[clave] = totales.get(clave, 0) + cantidad

# Mostrar una cabecera para facilitar la interpretación del resultado.
print("Tipo_analisis\tCategoria\tNumero_registros")

# Ordenar las categorías por tipo de análisis y, dentro de cada tipo,
# mostrar primero las que tienen mayor frecuencia.
for clave, cantidad in sorted(
    totales.items(),
    key=lambda elemento: (
        elemento[0].split("|", 1)[0],
        -elemento[1],
        elemento[0].split("|", 1)[1].casefold()
    )
):
    tipo, categoria = clave.split("|", 1)

    print(f"{tipo}\t{categoria}\t{cantidad}")

Overwriting reducer_meddocan.py


In [85]:
!hdfs dfs -rm -r -f /datasets/meddocan/resultados/analisis_edad_diagnosticos

!hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
  -files mapper_meddocan.py,reducer_meddocan.py \
  -mapper "python3 mapper_meddocan.py" \
  -reducer "python3 reducer_meddocan.py" \
  -input /datasets/meddocan/csv/meddocan_anonimizados.csv \
  -output /datasets/meddocan/resultados/analisis_edad_diagnosticos

Deleted /datasets/meddocan/resultados/analisis_edad_diagnosticos
packageJobJar: [/tmp/hadoop-unjar7867761578225036612/] [] /tmp/streamjob3500132230408501699.jar tmpDir=null
2026-10-29 09:42:41,953 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-10-29 09:42:42,246 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-10-29 09:42:42,624 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1792745247675_0018
2026-10-29 09:42:43,548 INFO mapred.FileInputFormat: Total input files to process : 1
2026-10-29 09:42:43,903 INFO mapreduce.JobSubmitter: number of splits:2
2026-10-29 09:42:44,261 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1792745247675_0018
2026-10-29 09:42:44,262 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-29 09:42:44,702 INFO conf.Configuration: resource-types.

In [86]:
!hdfs dfs -cat /datasets/meddocan/resultados/analisis_edad_diagnosticos/part-*

Tipo_analisis	Categoria	Numero_registros
Cirugía general y del Aparato digestivo	30_44	2
Neurología	45_59	2
Oncología	18_29	2
Oncología	30_44	2
SIN_ESPECIALIDAD	60_74	6
SIN_ESPECIALIDAD	30_44	3
SIN_ESPECIALIDAD	45_59	2
Urgencias	menor_18	2
Urologia	18_29	2
Urología	18_29	2
Urología	30_44	2
Urología	60_74	2
